# Tesla Stock Forecasting Using Machine Learning

Portfolio notebook for exploratory analysis, feature engineering, model comparison, and forecasting.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


## Load Dataset

In [ ]:
df = pd.read_csv('TESLA.csv', parse_dates=['Date'])
df.sort_values('Date', inplace=True)
df.head()

## Dataset Overview

In [ ]:
df.info()
df.describe()

In [ ]:
df.isnull().sum()

## Closing Price Trend

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(df['Date'], df['Close'])
plt.title('Tesla Closing Price History')
plt.show()

## Feature Engineering

In [ ]:
df['Lag_1'] = df['Close'].shift(1)
df['Lag_2'] = df['Close'].shift(2)
df['Lag_3'] = df['Close'].shift(3)

df['MA_5'] = df['Close'].rolling(5).mean()
df['MA_10'] = df['Close'].rolling(10).mean()
df['MA_20'] = df['Close'].rolling(20).mean()

df['Volatility'] = df['High'] - df['Low']
df.dropna(inplace=True)
df.head()

## Correlation Analysis

In [ ]:
corr = df.corr(numeric_only=True)

plt.figure(figsize=(10,8))
plt.imshow(corr, cmap='coolwarm', aspect='auto')
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.show()

## Model Training & Comparison

In [ ]:
features = ['Lag_1','Lag_2','Lag_3','MA_5','MA_10','MA_20','Volume','Volatility']

X = df[features]
y = df['Close']

split = int(len(df)*0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=300, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42)
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    results.append([
        name,
        mean_absolute_error(y_test, pred),
        np.sqrt(mean_squared_error(y_test, pred)),
        r2_score(y_test, pred)
    ])

results_df = pd.DataFrame(results, columns=['Model','MAE','RMSE','R2'])
results_df.sort_values('R2', ascending=False)

## Actual vs Predicted

In [ ]:
best_model = RandomForestRegressor(n_estimators=300, random_state=42)
best_model.fit(X_train, y_train)

best_predictions = best_model.predict(X_test)

plt.figure(figsize=(14,6))
plt.plot(y_test.index, y_test, label='Actual')
plt.plot(y_test.index, best_predictions, label='Predicted')
plt.legend()
plt.show()

## Feature Importance

In [ ]:
importance = pd.DataFrame({
    'Feature': features,
    'Importance': best_model.feature_importances_
}).sort_values('Importance', ascending=False)

importance

## Conclusion

- Built a time-series forecasting workflow.
- Compared multiple machine learning models.
- Evaluated performance using MAE, RMSE, and R².
- Analyzed feature importance and forecasting behavior.